# Phase 4 — Court Judgment Prediction — Model 7 — LoRMA

Run this notebook from top to bottom in a fresh kernel with the matching foundation checkpoint and tokenizer. The experiment is already selected in the configuration cell.

**Source:** `phase4/capstone-p4-all-models-all-methods.ipynb` at commit `7e7c8a9f1605c5f9836a3cb2cb7c28004343f5f3`. Model/training/evaluation/submission implementations are copied unchanged; only configuration selections and output directories differ. Saved outputs and execution counts were cleared during organization.

**Outputs:** use a separate Kaggle session or working directory for each run. The original submission filenames are preserved, so some ZIP files at the working-directory root can be overwritten by another run.

## Original implementation notes

# Phase 4 — Court Judgment Prediction (CJPE)
## Your own pre-trained models (Model 4 / Model 6 / Model 7) + Full / LoRA / LoRMA | Long-document hierarchical classifier

Same methodology as the InLegalBERT Phase 4 notebook, but the encoder is **one of your own Phase-1 models**. **No InLegalBERT anywhere.**

* **Select the architecture** with `MODEL_OPTION` in the config cell: `4` = Model 4 (`EfficientLegalLM`, 16 layers, 768d, GQA 12Q/2KV), `6` = Model 6 (`EfficientLegalLM`, 20 layers, 1,024d, GQA 16Q/4KV) or `7` = Model 7 (12-layer BERT-style stack, 768d, uncased WordPiece, prefix-LM pre-trained). Only the selected model is loaded.
* **Select the fine-tuning method(s)** with `RUN_METHODS`: `full`, `lora` (additive low-rank), `lorma` (multiplicative low-rank) or `kd` (knowledge distillation, as in your Phase 3 notebooks: the fully fine-tuned model is the teacher, a half-depth student initialised from evenly spaced pre-trained layers learns from the hard labels + the teacher's soft labels, and the student makes the submission) — run one or several.
* **Method (unchanged from Phase 4):** every case is one sample with one binary label. The whole judgment is tokenized once, up to `CHUNKS` windows are selected (`uniform` / `head_tail` / `sections`), all windows go through the encoder, window vectors are aggregated with learned attention (or mean) and a linear head predicts the outcome. Model selection, threshold tuning and the chunk ablation use only the **dev** split; test labels are never used.

**What necessarily differs from the InLegalBERT notebook (your models are not BERTs with a `[CLS]`):**
1. Models 4/6 are causal decoder-style LMs; as in your Phase 3 notebook they are run **bidirectionally** at fine-tune time (sliding-window layers use `BIDIR_WINDOW`) and have no `[CLS]` token, so the window vector is the **masked mean of the window's tokens** (`WINDOW_POOL='mean'`, also the default for Model 7, whose `[CLS]` was never trained by a classification objective). `WINDOW_POOL='first'` is available if you want to test the `[CLS]`/first-token variant.
2. Windows are wrapped with the model's own special tokens: `[CLS] … [SEP]` for Model 7, `… <eos>` for Models 4/6.
3. Learning rates default to the values you used for these models in Phase 3 (full 5e-5, adapters 3e-4, head 5e-4) instead of InLegalBERT's.

**Attach as Kaggle inputs:** the checkpoint dataset (`best.pt`) of the model you select and its tokenizer dataset (`tokenizer.json` for Models 4/6, `vocab.txt` or `tokenizer.json` for Model 7). Internet is only needed to download the competition data from the Hugging Face dataset repo.

**Caution (as in Phase 4):** legal judgments often contain the verdict explicitly. Decide whether outcome-bearing passages are allowed by your competition rules; if not, remove final-order/disposition sections consistently across train/dev/test.

## 1. Imports

In [ ]:
import os, json, math, random, re, gc, time, glob, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from tqdm.auto import tqdm
import transformers
from transformers import BertConfig, BertModel
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

## 2. Configuration — choose your model, fine-tuning method and document representation

In [ ]:
# ===================== WHICH OF YOUR MODELS =====================
MODEL_OPTION = 7                # <<< CHOOSE: 4 | 6 | 7
RUN_METHODS  = ['lorma']        # <<< 'full' | 'lora' | 'lorma' | 'kd' (one or several, e.g. ['full','kd'])

# ===================== document representation (as in the Phase 4 notebook) =====================
REPRESENTATION = 'head_tail'    # 'uniform' | 'head_tail' | 'sections'
POOLING        = 'attention'    # windows -> document: 'attention' | 'mean'
WINDOW_POOL    = 'mean'         # tokens -> window vector: 'mean' (masked mean) | 'first' (first token; [CLS] for Model 7)

# ===================== data / IO =====================
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
P4 = 'Phase_4_CJPE'
HF_TOKEN = os.environ.get('HF_TOKEN')   # falls back to the Kaggle secret "HF" in the data cell
CKPT_PATH = None                # set to force a specific best.pt (otherwise auto-found under /kaggle/input)
TOK_DIR   = None                # set to force a tokenizer folder (otherwise auto-found under /kaggle/input)
OUTPUT_ROOT = Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
OUTPUT_DIR  = OUTPUT_ROOT / 'cjpe_p4_model7_lorma'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===================== windows / training (Phase 4 structure) =====================
SEED=42; SMOKE=False; MAX_LEN=384; CHUNKS=6; STRIDE=256
BATCH_SIZE=1; GRAD_ACCUM=8; INFER_BATCH=1      # 1 case per micro-batch: encoder sees up to CHUNKS windows at once
EPOCHS=5; PATIENCE=2
HEAD_LR=5e-4; FULL_LR=5e-5; ADAPTER_LR=3e-4    # the values you used for these models in Phase 3
WEIGHT_DECAY=0.01; WARMUP_FRAC=0.08; CLASS_WEIGHT_POWER=0.5
RANK=8; ALPHA=16.0; DROPOUT=0.1
GRAD_CHECKPOINT=True            # recompute activations: needed for 6 windows/case on a T4 (applied to every method)
BIDIR_WINDOW=256                # Models 4/6: width of the sliding-window layers when run bidirectionally (as Phase 3)
NUM_WORKERS=0

# ===================== architecture of the selected model =====================
assert MODEL_OPTION in (4, 6, 7), "MODEL_OPTION must be 4, 6 or 7"
if MODEL_OPTION in (4, 6):
    ARCH = 'lm'                 # EfficientLegalLM (RMSNorm, RoPE, GQA, SwiGLU, alternating window/full layers)
    if MODEL_OPTION == 4:
        D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, D_FF, WINDOW_SIZE = 768, 16, 12, 2, 2048, 256
    else:
        D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, D_FF, WINDOW_SIZE = 1024, 20, 16, 4, 2432, 512
    HEAD_DIM, MAX_SEQ_LEN, ROPE_BASE = D_MODEL // N_HEADS, 1024, 10000.0
else:
    ARCH = 'bert'               # Model 7: BERT-base-style stack, prefix-LM pre-trained
    M7_CFG = dict(vocab_size=30522, hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
                  intermediate_size=3072, hidden_act='gelu', max_position_embeddings=512, type_vocab_size=2,
                  layer_norm_eps=1e-12, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1,
                  position_embedding_type='absolute')
    D_MODEL, MAX_SEQ_LEN = M7_CFG['hidden_size'], M7_CFG['max_position_embeddings']
    CKPT_KEY_PREFIX = ''        # optional extra prefix to strip from checkpoint keys (after '_orig_mod.' / 'module.' / 'bert.')
MODEL_NAME = f'Model {MODEL_OPTION}'

# ===================== knowledge distillation (method 'kd') =====================
# stage 1: the full-depth model is fully fine-tuned as the TEACHER (reused if 'full' is also in RUN_METHODS).
# stage 2: a half-depth STUDENT (initialised from evenly spaced pre-trained layers) is trained on
#          (1-KD_ALPHA) * class-weighted CE(hard labels) + KD_ALPHA * T^2 * KL(teacher soft labels || student).
KD_TEACHER_CKPT = None          # a best_model<N>_full_cjpe.pt from an earlier run -> skips teacher training
KD_TEMP, KD_ALPHA = 2.0, 0.5
NUM_LAYERS = N_LAYERS if ARCH == 'lm' else M7_CFG['num_hidden_layers']
KD_STUDENT_LAYERS = NUM_LAYERS // 2
KD_LAYER_MAP = [int(round(x)) for x in np.linspace(0, NUM_LAYERS - 1, KD_STUDENT_LAYERS)]
assert len(set(KD_LAYER_MAP)) == KD_STUDENT_LAYERS

assert all(m in ('full', 'lora', 'lorma', 'kd') for m in RUN_METHODS)
assert REPRESENTATION in ('uniform', 'head_tail', 'sections')
assert POOLING in ('attention', 'mean') and WINDOW_POOL in ('mean', 'first')
assert MAX_LEN <= MAX_SEQ_LEN, f'MAX_LEN {MAX_LEN} exceeds the context of {MODEL_NAME} ({MAX_SEQ_LEN})'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'; AMP_DTYPE = torch.float16
print(f'{MODEL_NAME} ({ARCH}) | d_model {D_MODEL} | methods {RUN_METHODS} | {REPRESENTATION}/{POOLING}/{WINDOW_POOL} | {DEVICE} | {OUTPUT_DIR}')

## 3. Locate the checkpoint and tokenizer of the selected model

In [ ]:
from transformers import PreTrainedTokenizerFast, BertTokenizerFast, AutoTokenizer

def _find(patterns, prefer=()):
    hits = sorted({h for p in patterns for h in glob.glob(p, recursive=True)})
    for key in prefer:
        pref = [h for h in hits if key in h.lower()]
        if pref:
            return pref[0], hits
    return (hits[0] if hits else None), hits

if CKPT_PATH is None:
    prefer = {4: ('m4',), 6: ('m6',), 7: ('m7', 'model7', 'model_7')}[MODEL_OPTION] + ('capstone',)
    CKPT_PATH, hits = _find(['/kaggle/input/**/best.pt'], prefer=prefer)
    print('best.pt candidates:', hits)
if TOK_DIR is None:
    if ARCH == 'lm':
        tj, hits = _find(['/kaggle/input/**/tokenizer.json'], prefer=('legal_tokenizer', 'tokenizer'))
    else:
        tj, hits = _find(['/kaggle/input/**/vocab.txt', '/kaggle/input/**/tokenizer.json'], prefer=('m7', 'model7', 'legal', 'tokenizer'))
    TOK_DIR = os.path.dirname(tj) if tj else None
    print('tokenizer candidates:', hits)
assert CKPT_PATH and os.path.isfile(CKPT_PATH), f'best.pt for {MODEL_NAME} not found - attach its checkpoint dataset or set CKPT_PATH'
assert TOK_DIR and os.path.isdir(TOK_DIR), f'tokenizer for {MODEL_NAME} not found - attach its tokenizer dataset or set TOK_DIR'
print(f'{MODEL_NAME}: make sure these belong to {MODEL_NAME}')
print('using checkpoint:', CKPT_PATH)
print('using tokenizer :', TOK_DIR)

if ARCH == 'lm':
    tokenizer = PreTrainedTokenizerFast.from_pretrained(TOK_DIR)
else:
    try:                                   # uncased WordPiece, 30,522 entries
        tokenizer = BertTokenizerFast.from_pretrained(TOK_DIR, do_lower_case=True)
    except Exception as e:
        print('BertTokenizerFast failed (', e, ') - falling back to AutoTokenizer')
        tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)
tokenizer.model_max_length = 10 ** 9      # we window ourselves; silence the length warning
VOCAB_SIZE = len(tokenizer)
print(f'VOCAB_SIZE={VOCAB_SIZE} fast={tokenizer.is_fast}')

if ARCH == 'lm':
    PAD_L = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    SEP_L = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else PAD_L
    CLS_L, N_SPECIAL = None, 1             # window = tokens + <eos>
    def wrap(chunk): return list(chunk) + [SEP_L]
else:
    assert VOCAB_SIZE == M7_CFG['vocab_size'], f'tokenizer has {VOCAB_SIZE} entries but Model 7 was built for {M7_CFG["vocab_size"]}'
    assert None not in (tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id)
    PAD_L, SEP_L, CLS_L, N_SPECIAL = tokenizer.pad_token_id, tokenizer.sep_token_id, tokenizer.cls_token_id, 2
    def wrap(chunk): return [CLS_L] + list(chunk) + [SEP_L]     # window = [CLS] tokens [SEP]
    print('tokenizer lowercases:', tokenizer.tokenize('Section 302 IPC, The Supreme Court'))
print('PAD', PAD_L, 'SEP/EOS', SEP_L, 'CLS', CLS_L, '| special tokens per window:', N_SPECIAL)

## 4. Download Phase 4 data and validate schema (unchanged from the InLegalBERT notebook)

In [ ]:
# === Phase 4: Load and Normalize Train / Dev / Test ===

import json
import numpy as np
from collections import Counter

# For private HF datasets, set HF_TOKEN environment variable
# or create Kaggle secret named HF.
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF")
    except Exception:
        pass


def fetch(rel):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        filename=rel,
        token=HF_TOKEN
    )


def normalize(raw, labeled, split_name="unknown"):

    # Handle different JSON structures
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list):
            raw = raw["data"]
        elif isinstance(raw.get("examples"), list):
            raw = raw["examples"]
        else:
            raw = [
                dict(v, id=k) if isinstance(v, dict)
                else {"id": k, "text": v}
                for k, v in raw.items()
            ]

    assert isinstance(raw, list), (
        "Expected list of records or id-indexed dictionary"
    )

    out = []

    # === Step 1: Normalize records ===
    for row in raw:

        assert (
            isinstance(row, dict)
            and "id" in row
            and "text" in row
        ), f"Bad row: {str(row)[:200]}"

        text = row["text"]

        if isinstance(text, list):
            text = "\n".join(map(str, text))

        assert isinstance(text, str), (
            f'Unexpected text for {row["id"]}'
        )

        r = {
            "id": str(row["id"]),
            "text": text
        }

        if labeled:
            keys = [
                k for k in (
                    "label",
                    "labels",
                    "outcome",
                    "judgment",
                    "target"
                )
                if k in row
            ]

            assert keys, (
                f'Missing label for {r["id"]}, '
                f"available keys {list(row)}"
            )

            y = row[keys[0]]

            if isinstance(y, list) and len(y) == 1:
                y = y[0]

            assert str(y) in ("0", "1"), (
                f'Nonbinary label {y!r} on {r["id"]}'
            )

            r["label"] = int(y)

        out.append(r)

    # === Step 2: Detect duplicate IDs ===
    ids = [r["id"] for r in out]
    counts = Counter(ids)

    duplicates = {
        k: v for k, v in counts.items()
        if v > 1
    }

    print(f"\n{'=' * 55}")
    print(f"Split: {split_name}")
    print(f"Original records: {len(out):,}")
    print(f"Unique case IDs: {len(counts):,}")
    print(f"Duplicated case IDs: {len(duplicates):,}")
    print(
        "Extra duplicate records:",
        sum(v - 1 for v in duplicates.values())
    )

    if duplicates:
        print(
            "Sample duplicate IDs:",
            list(duplicates.items())[:10]
        )

    # === Step 3: Check conflicting labels ===
    if labeled:
        id_labels = {}

        for r in out:
            id_labels.setdefault(
                r["id"], set()
            ).add(r["label"])

        conflicts = {
            k: v for k, v in id_labels.items()
            if len(v) > 1
        }

        print(
            f"IDs with conflicting labels: {len(conflicts)}"
        )

        if conflicts:
            print(
                "Sample conflicting IDs:",
                list(conflicts.items())[:10]
            )

            raise ValueError(
                f"{split_name} contains identical case IDs "
                "with conflicting labels. Inspect these records "
                "before proceeding."
            )

    # === Step 4: Merge records with identical case IDs ===
    merged = {}
    seen_texts = {}

    exact_duplicate_records = 0
    merged_text_segments = 0

    for r in out:
        case_id = r["id"]
        text = r["text"]

        if case_id not in merged:
            merged[case_id] = r.copy()
            seen_texts[case_id] = {text}

        else:
            # Do not repeat identical text segments
            if text in seen_texts[case_id]:
                exact_duplicate_records += 1
                continue

            # Combine different text segments of the same case
            if text.strip():
                if merged[case_id]["text"].strip():
                    merged[case_id]["text"] += "\n\n" + text
                else:
                    merged[case_id]["text"] = text

                merged_text_segments += 1

            seen_texts[case_id].add(text)

    out = list(merged.values())

    print(f"Exact duplicate texts skipped: {exact_duplicate_records:,}")
    print(f"Additional text segments merged: {merged_text_segments:,}")
    print(f"Final cases: {len(out):,}")
    print(f"{'=' * 55}")

    # === Step 5: Verify uniqueness ===
    final_ids = [r["id"] for r in out]

    assert len(final_ids) == len(set(final_ids)), (
        f"Duplicate IDs remain in {split_name}"
    )

    return out


def read_split(name):

    folder = "Test" if name == "test" else "Train"

    filepath = fetch(
        f"{P4}/{folder}/{name}.json"
    )

    with open(filepath, encoding="utf-8") as f:
        raw = json.load(f)

    return normalize(
        raw,
        labeled=(name != "test"),
        split_name=name
    )


# === Load all splits ===
train = read_split("train")
dev = read_split("dev")
test = read_split("test")


# === Verify train/dev separation ===
train_ids = {x["id"] for x in train}
dev_ids = {x["id"] for x in dev}

overlap = train_ids & dev_ids

print(f"\nTrain/Dev overlapping IDs: {len(overlap)}")

assert not overlap, (
    f"Train and dev share {len(overlap)} case IDs. "
    "This could cause data leakage."
)


# === Dataset Statistics ===
print("\n" + "=" * 55)
print("FINAL DATASET STATISTICS")
print("=" * 55)

for name, arr in [
    ("train", train),
    ("dev", dev),
    ("test", test)
]:

    lengths = [len(r["text"]) for r in arr]

    print(f"\n{name.upper()}")
    print(f"Cases: {len(arr):,}")
    print(f"Median text length: {int(np.median(lengths)):,} chars")
    print(f"Mean text length: {int(np.mean(lengths)):,} chars")
    print(f"Max text length: {max(lengths):,} chars")

    if name != "test":
        class_counts = Counter(
            r["label"] for r in arr
        )
        print("Class counts:", dict(class_counts))


# === Smoke Test ===
if SMOKE:
    train = train[:24]
    dev = dev[:12]
    test = test[:12]
    EPOCHS = 1

    print("\nSMOKE MODE ENABLED")


# === Final Checks ===
assert len(train) and len(dev) and len(test), (
    "One or more splits are empty"
)

print("\nSchema example:")
print({
    "id": train[0]["id"],
    "text_chars": len(train[0]["text"]),
    "label": train[0]["label"]
})

print("\nDataset loading completed successfully!")

## 5. Document window selection (unchanged logic)
Tokenize the **whole document once** (without truncation), then sample at most `CHUNKS` windows.

- `uniform`: evenly spaced windows across full text.
- `head_tail`: keeps the beginning and end, spreads the rest through the middle (implemented exactly as in the Phase 4 notebook).
- `sections`: heuristic header-based selection for facts/arguments/reasoning/orders, with fallback to head/tail.

Each window is wrapped with the selected model's special tokens (`[CLS] … [SEP]` for Model 7, `… <eos>` for Models 4/6).

In [ ]:
WINDOW_TOKENS=MAX_LEN-N_SPECIAL
assert 16<STRIDE<=WINDOW_TOKENS
HEADER_RE=re.compile(r'(?im)^\s*(?:\d+[.)]\s*)?(facts?|background|brief facts|issues?|arguments?|submissions?|reasoning|analysis|discussion|findings?|conclusion|final order|order|judg(?:e)?ment)\s*[:.\-]?\s*$')

def make_starts(n, mode):
    if n<=WINDOW_TOKENS:return [0]
    max_start=n-WINDOW_TOKENS
    all_starts=list(range(0,max_start+1,STRIDE))
    if all_starts[-1]!=max_start:all_starts.append(max_start)
    if len(all_starts)<=CHUNKS:return all_starts
    if mode=='uniform':
        inds=np.linspace(0,len(all_starts)-1,CHUNKS).round().astype(int)
        return [all_starts[i] for i in inds]
    # start/end plus regular middle coverage
    inds=np.linspace(0,len(all_starts)-1,CHUNKS).round().astype(int)
    return [all_starts[i] for i in inds]

def windows_for(text, mode=REPRESENTATION):
    ids=tokenizer(text,add_special_tokens=False,truncation=False)['input_ids']
    if not ids: ids=[tokenizer.unk_token_id if tokenizer.unk_token_id is not None else PAD_L]
    if mode=='sections':
        # Find section header offsets using tokenizer over each preceding prefix; costly but done once in preprocessing.
        matches=list(HEADER_RE.finditer(text))
        anchors=[]
        for match in matches[:40]:
            offset=len(tokenizer(text[:match.start()],add_special_tokens=False)['input_ids'])
            anchors.append(max(0,min(offset,len(ids)-WINDOW_TOKENS)))
        desired=[0,max(0,len(ids)-WINDOW_TOKENS)]+anchors
        # Cover document even when few headers exist
        desired+=make_starts(len(ids),'head_tail')
        chosen=[]
        for v in desired:
            if not chosen or all(abs(v-a)>=WINDOW_TOKENS//3 for a in chosen):chosen.append(v)
            if len(chosen)>=CHUNKS:break
        starts=sorted(chosen)
    else:starts=make_starts(len(ids),mode)
    chunks=[]
    for st in starts[:CHUNKS]:
        chunk=ids[st:st+WINDOW_TOKENS]
        chunks.append(wrap(chunk))
    return chunks or [wrap(ids[:WINDOW_TOKENS])]

class CaseDataset(Dataset):
    def __init__(self,records,mode):
        self.rows=records; self.data=[]
        for r in tqdm(records,desc=f'Windowize {mode}'):
            self.data.append(windows_for(r['text'],mode))
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):return self.data[i], self.rows[i].get('label',-1),self.rows[i]['id']

def collate(batch):
    B=len(batch); C=max(len(z[0]) for z in batch); L=max(len(w) for z in batch for w in z[0])
    ids=torch.full((B,C,L),PAD_L,dtype=torch.long)
    mask=torch.zeros((B,C,L),dtype=torch.long)
    cm=torch.zeros((B,C),dtype=torch.bool)
    for b,(wins,_,_) in enumerate(batch):
        for c,w in enumerate(wins):ids[b,c,:len(w)]=torch.tensor(w);mask[b,c,:len(w)]=1;cm[b,c]=True
    ys=torch.tensor([z[1] for z in batch],dtype=torch.long)
    return ids,mask,cm,ys,[z[2] for z in batch]

train_ds=CaseDataset(train,REPRESENTATION);dev_ds=CaseDataset(dev,REPRESENTATION);test_ds=CaseDataset(test,REPRESENTATION)
print('Example # chunks:',len(train_ds.data[0]),'first chunk tokens',len(train_ds.data[0][0]))

## 6. Your pre-trained backbones — definitions and checkpoint loading
Definitions are copied from your Phase 3 notebooks (`EfficientLegalLM` for Models 4/6; the BERT-style loader for Model 7). Only the selected one is instantiated.

### 6a. Models 4 / 6 — `EfficientLegalLM` (from `capstone-p3-model-4-and-6.ipynb`)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).type_as(x) * self.weight


def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, freqs_cos, freqs_sin):
    x_r = x.reshape(*x.shape[:-1], -1, 2)
    x0, x1 = x_r[..., 0], x_r[..., 1]
    seq_len = x.shape[2]
    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack([out0, out1], dim=-1).reshape_as(x)


class SwiGLU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads
        self.use_window = use_window; self.window_size = window_size
        self.attn_p = dropout
        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, freqs_cos, freqs_sin)
        k = apply_rope(k, freqs_cos, freqs_sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v) if use_cache else None
        dp = self.attn_p if self.training else 0.0
        if past_kv is not None:
            if self.use_window:
                k = k[:, :, -self.window_size:]; v = v[:, :, -self.window_size:]
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        else:
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            if self.use_window and T > 1:
                row = torch.arange(T, device=x.device).unsqueeze(1)
                col = torch.arange(T, device=x.device).unsqueeze(0)
                mask = (col <= row) & (col > row - self.window_size)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)
            else:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.out_proj(out)), new_kv


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        attn_out, new_kv = self.attn(self.attn_norm(x), freqs_cos, freqs_sin, use_cache, past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv


class EfficientLegalLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff,
                 max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True):
        super().__init__()
        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len; self.use_gc = use_gc
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,
                             use_window=(i % 2 == 0), window_size=window_size)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)
        self.register_buffer("freqs_cos", fc, persistent=False)
        self.register_buffer("freqs_sin", fs, persistent=False)

    def forward(self, input_ids, targets=None, use_cache=False, past_kvs=None):
        B, T = input_ids.shape
        x = self.emb_dropout(self.tok_emb(input_ids))
        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0
        fc = self.freqs_cos[past_len:past_len + T]
        fs = self.freqs_sin[past_len:past_len + T]
        presents = [] if use_cache else None
        for i, layer in enumerate(self.layers):
            pkv = past_kvs[i] if past_kvs is not None else None
            x, kv = layer(x, fc, fs, use_cache, pkv)
            if use_cache:
                presents.append(kv)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits[:, :-1, :].contiguous().view(-1, self.vocab_size),
                                   targets[:, 1:].contiguous().view(-1), ignore_index=-100)
        return logits, loss, presents

### 6b. Model 7 — checkpoint helpers (from `capstone-p3-model7.ipynb`)

In [ ]:
from transformers import BertConfig, BertModel


def build_model7():
    """Model 7 backbone: embeddings (token + learned position + segment, LayerNorm) -> 12 post-LN layers -> pooler (unused)."""
    return BertModel(BertConfig(**M7_CFG), add_pooling_layer=True)


def _norm_key(k):
    for pre in ("_orig_mod.", "module."):
        while k.startswith(pre):
            k = k[len(pre):]
    if CKPT_KEY_PREFIX and k.startswith(CKPT_KEY_PREFIX):
        k = k[len(CKPT_KEY_PREFIX):]
    return k.replace(".LayerNorm.gamma", ".LayerNorm.weight").replace(".LayerNorm.beta", ".LayerNorm.bias")


def split_checkpoint(raw):
    """Return (encoder_state_dict, prediction_head_state_dict) from a Model 7 checkpoint of any common wrapping."""
    sd = raw
    if isinstance(raw, dict):
        for k in ("model_state_dict", "model", "state_dict", "global_state_dict", "theta_g"):
            if k in raw and isinstance(raw[k], dict):
                sd = raw[k]; break
    enc, head = {}, {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        k = _norm_key(k)
        if k.startswith("bert."):
            enc[k[len("bert."):]] = v
        elif k.startswith("cls."):
            head[k] = v
        else:
            enc[k] = v
    return enc, head


class PredHead:
    """BERT prediction (LM) head used only for the load sanity check: dense -> GELU -> LayerNorm -> tied decoder (+ bias)."""
    def __init__(self, head_sd, emb_w):
        g = lambda *names: next((head_sd[n] for n in names if n in head_sd), None)
        self.dw = g("cls.predictions.transform.dense.weight"); self.db = g("cls.predictions.transform.dense.bias")
        self.lw = g("cls.predictions.transform.LayerNorm.weight"); self.lb = g("cls.predictions.transform.LayerNorm.bias")
        self.bias = g("cls.predictions.bias", "cls.predictions.decoder.bias")
        self.emb_w = emb_w
        self.ok = None not in (self.dw, self.db, self.lw, self.lb)

    def __call__(self, h):
        d = F.gelu(F.linear(h, self.dw.to(h.device, h.dtype), self.db.to(h.device, h.dtype)))
        d = F.layer_norm(d, (D_MODEL,), self.lw.to(h.device, h.dtype), self.lb.to(h.device, h.dtype), M7_CFG["layer_norm_eps"])
        b = self.bias.to(h.device, h.dtype) if self.bias is not None else None
        return F.linear(d, self.emb_w.to(h.dtype), b)

### 6c. Load the selected model (with the same sanity checks as Phase 3)

In [ ]:
raw_ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
if ARCH == 'lm':
    ENC_SD = {(k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k): v for k, v in raw_ckpt['model_state_dict'].items()}
    print('checkpoint meta:', {k: raw_ckpt.get(k) for k in ('step', 'loss')})
    HEAD_SD = None
else:
    ENC_SD, HEAD_SD = split_checkpoint(raw_ckpt)
    print('checkpoint meta:', {k: raw_ckpt[k] for k in ('step', 'round', 'loss', 'val_loss') if isinstance(raw_ckpt, dict) and k in raw_ckpt})
del raw_ckpt

def build_backbone():
    # a fresh copy of the pre-trained model, weights loaded from ENC_SD (checked every time)
    if ARCH == 'lm':
        b = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                             MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
        missing, unexpected = b.load_state_dict(ENC_SD, strict=False)
        missing = [k for k in missing if k != 'lm_head.weight']          # tied to tok_emb.weight
    else:
        b = build_model7()
        missing, unexpected = b.load_state_dict(ENC_SD, strict=False)
        missing = [k for k in missing if not k.startswith('pooler.')]    # never trained by the continuation loss
        unexpected = [k for k in unexpected if k != 'embeddings.position_ids']
    assert not missing and not unexpected, f'checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}'
    return b

# ---- sanity: a correctly loaded model is far better than chance on a training judgment ----
probe = build_backbone().to(DEVICE).eval()
_ids = tokenizer(train[0]['text'], add_special_tokens=False)['input_ids']
if ARCH == 'lm':
    ids0 = torch.tensor([_ids[:512]], device=DEVICE)
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
        lm_loss = probe(ids0, targets=ids0)[1].item()
    print(f'LM sanity loss on a training judgment: {lm_loss:.3f}  (expect roughly 2-4; chance = {math.log(VOCAB_SIZE):.1f})')
    assert lm_loss < 6.0, 'LM loss is too high - weights or tokenizer do not match'
else:
    ph = PredHead(dict(HEAD_SD), probe.embeddings.word_embeddings.weight.detach())
    if ph.ok:
        ids_ = _ids[:509]
        P = min(320, int(0.63 * len(ids_))); S = len(ids_) - P
        seq = [CLS_L] + ids_[:P] + [SEP_L] + ids_[P:] + [SEP_L]
        T, Lp = len(seq), P + 2
        idx = torch.arange(T, device=DEVICE)
        pm = ((idx[None, :] < Lp) | (idx[None, :] <= idx[:, None])).long()[None]     # prefix bidirectional, suffix causal
        tt = torch.tensor([[0] * Lp + [1] * (T - Lp)], device=DEVICE)
        try:
            with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                hs = probe(input_ids=torch.tensor([seq], device=DEVICE), attention_mask=pm, token_type_ids=tt).last_hidden_state
                lg = ph(hs[:, Lp - 1:T - 1])
            tgt = torch.tensor([seq[Lp:]], device=DEVICE)
            lm_loss = F.cross_entropy(lg.float().reshape(-1, VOCAB_SIZE), tgt.reshape(-1)).item()
            print(f'continuation sanity loss ({S} suffix tokens): {lm_loss:.3f}  (chance = {math.log(VOCAB_SIZE):.2f}; a trained model is clearly lower)')
            assert lm_loss < 8.0, 'continuation loss is near chance - weights or tokenizer do not match'
        except AssertionError:
            raise
        except Exception as e:
            print('sanity check skipped (3-D attention mask not supported by this transformers version):', repr(e)[:200])
    else:
        print('prediction-head weights not in the checkpoint - sanity check skipped (only the encoder is needed)')
    del ph
del probe, HEAD_SD, _ids
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print(f'{MODEL_NAME} loaded OK: {sum(v.numel() for v in ENC_SD.values()) / 1e6:.1f}M tensor elements in the checkpoint')

## 7. Low-rank adapters and the hierarchical classifier
`LoRALinear` / `LoRMALinear` are the Phase 4 definitions (LoRA: `Wx + α/r·B(Ax)`; LoRMA: multiplicative `(I + α/r·BA)Wx`). They wrap every linear inside the transformer layers of the selected model (Models 4/6: q/k/v/out + SwiGLU gate/up/down; Model 7: q/k/v/attention-out + FFN up/down). The hierarchical head is the Phase 4 one: window vectors → attention/mean over windows → linear classifier.

In [ ]:
class LoRALinear(nn.Module):
    # Conventional additive update: Wx + alpha/r * B(Ax).
    def __init__(self,base,rank,alpha):
        super().__init__();self.base=base
        self.A=nn.Parameter(torch.empty(rank,base.in_features))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5));self.scale=alpha/rank
    def forward(self,x):
        return self.base(x)+self.scale*F.linear(F.linear(x,self.A),self.B)

class LoRMALinear(nn.Module):
    # SAME definition as Phase-3: multiplicative output-space adaptation
    # (I + alpha/r * B A) W x, frozen W, B starts zero.
    def __init__(self,base,rank,alpha):
        super().__init__();self.base=base
        self.A=nn.Parameter(torch.randn(rank,base.out_features)*0.02)
        self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        self.scale=alpha/rank
    def forward(self,x):
        h=self.base(x);return h+self.scale*F.linear(F.linear(h,self.A),self.B)

def add_adapters(backbone, method):
    assert method in ('lora', 'lorma')
    wrapper = LoRALinear if method == 'lora' else LoRMALinear
    count = 0
    if ARCH == 'lm':
        for blk in backbone.layers:
            for parent in (blk.attn, blk.ffn):
                for name, child in list(parent.named_children()):
                    if isinstance(child, nn.Linear):
                        setattr(parent, name, wrapper(child, RANK, ALPHA)); count += 1
    else:
        for blk in backbone.encoder.layer:
            targets = [(parent, name, child) for parent in blk.modules()
                       for name, child in parent.named_children() if isinstance(child, nn.Linear)]
            for parent, name, child in targets:
                setattr(parent, name, wrapper(child, RANK, ALPHA)); count += 1
    return count

# ---- bidirectional attention for Models 4/6 (every query may always attend to itself, so padded rows never become NaN)
def build_masks(T, pad_mask, window):
    r = torch.arange(T, device=pad_mask.device).unsqueeze(1)
    c = torch.arange(T, device=pad_mask.device).unsqueeze(0)
    eye = torch.eye(T, dtype=torch.bool, device=pad_mask.device)[None, None]
    key = pad_mask[:, None, None, :]
    full = torch.ones(T, T, dtype=torch.bool, device=pad_mask.device)
    win = (c - r).abs() < window
    return (full[None, None] & key) | eye, (win[None, None] & key) | eye

def block_forward(blk, x, cos, sin, mask):
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, _ = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)
    k = repeat_kv(k, a.n_rep)
    v = repeat_kv(v, a.n_rep)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, -1)
    x = x + a.out_proj(out)
    return x + blk.ffn(blk.ffn_norm(x))

class WindowEncoder(nn.Module):
    # (N, L) token ids + pad mask  ->  (N, L, D) hidden states, for either architecture
    def __init__(self, backbone, use_gc):
        super().__init__()
        self.backbone, self.use_gc = backbone, use_gc
        if ARCH == 'bert' and use_gc:
            backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    def forward(self, ids, mask):
        m = self.backbone
        if ARCH == 'bert':      # whole window is the bidirectional prefix: segment id 0 everywhere
            return m(input_ids=ids, attention_mask=mask.long(), token_type_ids=torch.zeros_like(ids)).last_hidden_state
        T = ids.size(1)
        x = m.tok_emb(ids)
        cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
        full, win = build_masks(T, mask.bool(), BIDIR_WINDOW)
        for blk in m.layers:
            mk = win if blk.attn.use_window else full
            if self.use_gc and self.training:
                x = grad_checkpoint(block_forward, blk, x, cos, sin, mk, use_reentrant=False)
            else:
                x = block_forward(blk, x, cos, sin, mk)
        return m.final_norm(x)

def pool_windows(h, mask, how):
    if how == 'first':
        return h[:, 0].float()
    m = mask.unsqueeze(-1).float()
    return (h.float() * m).sum(1) / m.sum(1).clamp(min=1)

class CJPEClassifier(nn.Module):
    def __init__(self, backbone, pooling, window_pool, use_gc):
        super().__init__()
        self.encoder = WindowEncoder(backbone, use_gc)
        self.pooling, self.window_pool = pooling, window_pool
        h = D_MODEL
        self.chunk_att = nn.Sequential(nn.Linear(h, h // 2), nn.Tanh(), nn.Linear(h // 2, 1))
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(h, 2))
    def forward(self, ids, mask, cm, chunk_selection=None):
        B, C, L = ids.shape
        valid = cm.reshape(-1)
        # Skip padded chunks entirely.
        xi = ids.reshape(B * C, L)[valid]; xm = mask.reshape(B * C, L)[valid]
        hs = pool_windows(self.encoder(xi, xm), xm, self.window_pool)
        embeddings = hs.new_zeros(B * C, hs.size(-1)); embeddings[valid] = hs
        embeddings = embeddings.reshape(B, C, -1)
        allowed = cm.clone()
        if chunk_selection in ('first', 'middle', 'last'):
            allowed[:] = False
            for b in range(B):
                idx = torch.where(cm[b])[0]
                if len(idx):
                    pos = {'first': 0, 'middle': len(idx) // 2, 'last': -1}[chunk_selection]
                    allowed[b, idx[pos]] = True
        if self.pooling == 'mean':
            weights = allowed.float() / allowed.sum(-1, keepdim=True).clamp(min=1)
        else:
            scores = self.chunk_att(embeddings).squeeze(-1).masked_fill(~allowed, -1e4)
            weights = torch.softmax(scores, dim=-1)
        doc = (embeddings * weights.unsqueeze(-1)).sum(dim=1)
        return self.head(doc), weights

def build_student():
    # half-depth copy: embeddings / final norm copied, layer i <- pre-trained layer KD_LAYER_MAP[i]
    if ARCH == 'lm':
        st = EfficientLegalLM(VOCAB_SIZE, D_MODEL, len(KD_LAYER_MAP), N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                              MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
        pre = 'layers.'
    else:
        st = BertModel(BertConfig(**{**M7_CFG, 'num_hidden_layers': len(KD_LAYER_MAP)}), add_pooling_layer=True)
        pre = 'encoder.layer.'
    new = {k: v for k, v in ENC_SD.items() if not k.startswith(pre)}
    for i, src in enumerate(KD_LAYER_MAP):
        p = f'{pre}{src}.'
        for k, v in ENC_SD.items():
            if k.startswith(p): new[f'{pre}{i}.' + k[len(p):]] = v
    missing, unexpected = st.load_state_dict(new, strict=False)
    missing = [k for k in missing if k != 'lm_head.weight' and not k.startswith('pooler.')]
    unexpected = [k for k in unexpected if k != 'embeddings.position_ids']
    assert not missing and not unexpected, (missing[:5], unexpected[:5])
    if ARCH == 'lm':
        for i, src in enumerate(KD_LAYER_MAP): st.layers[i].attn.use_window = (src % 2 == 0)   # same attention type as the source layer
    return st

def make_model(method):
    base = build_student() if method == 'kd' else build_backbone()
    if method in ('full', 'kd'):
        for p in base.parameters(): p.requires_grad_(True)
        if ARCH == 'bert':
            for p in base.pooler.parameters(): p.requires_grad_(False)      # unused by the classifier
    else:
        for p in base.parameters(): p.requires_grad_(False)
        print(method, 'adapted linear layers:', add_adapters(base, method))
    model = CJPEClassifier(base, POOLING, WINDOW_POOL, GRAD_CHECKPOINT).to(DEVICE)
    print(f'{MODEL_NAME} {method} trainable / total:', sum(p.numel() for p in model.parameters() if p.requires_grad),
          '/', sum(p.numel() for p in model.parameters()))
    return model

## 8. Training, case-level evaluation, checkpoint selection (same procedure as Phase 4)

In [ ]:
from transformers import get_linear_schedule_with_warmup

def loader(ds,shuffle=False,batch=None):
    return DataLoader(ds,batch_size=batch or BATCH_SIZE,shuffle=shuffle,collate_fn=collate,num_workers=NUM_WORKERS)

def move(batch):
    ids,mask,cm,y,keys=batch
    return ids.to(DEVICE),mask.to(DEVICE),cm.to(DEVICE),y.to(DEVICE),keys

@torch.no_grad()
def predict_prob(model,ds,selection=None):
    model.eval(); all_probs=[];all_y=[];all_ids=[]
    for b in tqdm(loader(ds,batch=INFER_BATCH),desc='Inference',leave=False):
        x,mask,cm,y,keys=move(b)
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=AMP):
            logit,_=model(x,mask,cm,selection)
        all_probs.extend(logit.float().softmax(-1)[:,1].cpu().numpy().tolist())
        all_y.extend(y.cpu().numpy().tolist());all_ids.extend(keys)
    return np.array(all_probs),np.array(all_y),all_ids

def metrics(y,p,t=0.5):
    pred=(p>=t).astype(int)
    return {'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0)),
            'macro_precision':float(precision_score(y,pred,average='macro',zero_division=0)),
            'macro_recall':float(recall_score(y,pred,average='macro',zero_division=0))}

def best_threshold(y,prob):
    # Tune solely on dev; fixed 0.5 also reported to reveal threshold effects.
    grid=np.linspace(.1,.9,81)
    ranked=sorted([(metrics(y,prob,t)['macro_f1'],-abs(t-.5),t) for t in grid],reverse=True)
    return float(ranked[0][2])

@torch.no_grad()
def predict_logits(model, ds):
    model.eval(); out = {}
    for b in tqdm(loader(ds, batch=INFER_BATCH), desc='Logits', leave=False):
        x, mask, cm, y, keys = move(b)
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=AMP):
            logit, _ = model(x, mask, cm)
        for k, l in zip(keys, logit.float().cpu().numpy()): out[k] = l
    return out

def trainable_state(model):
    # Everything not listed here is identical to the pre-trained checkpoint, which make_model() reloads.
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def fit(method, teacher_logits=None):
    model = make_model(method)
    group_enc = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('encoder.')]
    group_head = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('encoder.')]
    lr = FULL_LR if method in ('full', 'kd') else ADAPTER_LR
    opt = torch.optim.AdamW([{'params': group_enc, 'lr': lr}, {'params': group_head, 'lr': HEAD_LR}], weight_decay=WEIGHT_DECAY)
    total_steps = math.ceil(len(loader(train_ds, True)) / GRAD_ACCUM) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(total_steps * WARMUP_FRAC), total_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    freq = np.bincount([r['label'] for r in train], minlength=2).astype(float) + 1
    w = 1 / freq ** CLASS_WEIGHT_POWER; w /= np.dot(w, freq) / sum(freq)
    class_w = torch.tensor(w, device=DEVICE, dtype=torch.float32)
    best = -1; bad = 0; path = OUTPUT_DIR / f'best_model{MODEL_OPTION}_{method}_cjpe.pt'
    for epoch in range(EPOCHS):
        model.train(); opt.zero_grad(set_to_none=True); losses = []
        batches = loader(train_ds, shuffle=True)
        for i, batch in enumerate(tqdm(batches, desc=f'{MODEL_NAME} {method} epoch {epoch + 1}/{EPOCHS}', leave=False)):
            x, am, cm, y, keys = move(batch)
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=AMP):
                logit, _ = model(x, am, cm)
                loss = F.cross_entropy(logit.float(), y, weight=class_w)
                if teacher_logits is not None:      # knowledge distillation: blend with the teacher's soft labels
                    tl = torch.tensor(np.stack([teacher_logits[k] for k in keys]), device=DEVICE, dtype=torch.float32)
                    kd = F.kl_div(F.log_softmax(logit.float() / KD_TEMP, -1), F.softmax(tl / KD_TEMP, -1), reduction='batchmean') * KD_TEMP * KD_TEMP
                    loss = (1.0 - KD_ALPHA) * loss + KD_ALPHA * kd
            # Divide by actual group size on final partial accumulation window.
            group_size = min(GRAD_ACCUM, len(batches) - (i // GRAD_ACCUM) * GRAD_ACCUM)
            scaler.scale(loss / group_size).backward(); losses.append(loss.item())
            if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(batches):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        probs, y, _ = predict_prob(model, dev_ds)
        m = metrics(y, probs, .5)
        print(MODEL_NAME, method, 'epoch', epoch + 1, 'train_loss', round(np.mean(losses), 4), 'dev @ .5', m)
        if m['macro_f1'] > best + 1e-6:
            best = m['macro_f1']; bad = 0
            torch.save({'state': trainable_state(model), 'method': method, 'model_option': MODEL_OPTION,
                        'representation': REPRESENTATION, 'pooling': POOLING, 'window_pool': WINDOW_POOL,
                        'rank': RANK, 'alpha': ALPHA, 'max_len': MAX_LEN, 'chunks': CHUNKS, 'stride': STRIDE,
                        'best_dev_f1_05': best}, path)
        else:
            bad += 1
            if bad >= PATIENCE: print('Early stopping'); break
    return model, path

def reload_best(model, path):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    assert payload['model_option'] == MODEL_OPTION, f"checkpoint is for Model {payload['model_option']}, notebook is set to Model {MODEL_OPTION}"
    state = payload['state']
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert not unexpected, unexpected[:5]
    expected = {n for n, p in model.named_parameters() if p.requires_grad}
    assert expected <= set(state), list(expected - set(state))[:5]
    return model

## 9. Run the selected method(s); compare dev Macro-F1

In [ ]:
results = {}; predictions = {}; training_summary = []

def teacher_train_logits():
    # KD stage 1: full-depth teacher (reuses the 'full' run of this session, or KD_TEACHER_CKPT, otherwise trains it now)
    tpath = Path(KD_TEACHER_CKPT) if KD_TEACHER_CKPT else OUTPUT_DIR / f'best_model{MODEL_OPTION}_full_cjpe.pt'
    if not KD_TEACHER_CKPT and 'full' not in results:
        print('KD stage 1: fine-tuning the full-depth teacher')
        t, _ = fit('full'); del t; gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    teacher = reload_best(make_model('full'), tpath).to(DEVICE)
    pt, yt, _ = predict_prob(teacher, dev_ds)
    print('teacher dev @ .5:', metrics(yt, pt, .5))
    logits = predict_logits(teacher, train_ds)
    del teacher; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return logits

for method in sorted(RUN_METHODS, key=lambda m: m == 'kd'):      # 'full' first so 'kd' can reuse it as teacher
    print(f'\n=== {MODEL_NAME} | {method.upper()} ===')
    if method == 'kd':
        print(f'KD student: {len(KD_LAYER_MAP)} layers copied from pre-trained layers {KD_LAYER_MAP}')
        model, path = fit('kd', teacher_logits=teacher_train_logits())
    else:
        model, path = fit(method)
    model = reload_best(model, path).to(DEVICE)
    pdev, ydev, dev_ids = predict_prob(model, dev_ds)
    th = best_threshold(ydev, pdev)
    ptest, _, test_ids = predict_prob(model, test_ds)
    assert test_ids == [r['id'] for r in test]
    res = {'model': MODEL_NAME, 'method': method, 'representation': REPRESENTATION, 'pooling': POOLING,
           'window_pool': WINDOW_POOL, 'dev_f1_05': metrics(ydev, pdev, .5)['macro_f1'],
           'threshold': th, **{'dev_' + k: v for k, v in metrics(ydev, pdev, th).items()}}
    results[method] = res
    predictions[method] = {'dev_prob': pdev, 'ydev': ydev, 'test_prob': ptest, 'dev_ids': dev_ids,
                           'test_ids': test_ids, 'threshold': th}
    print(res)
    training_summary.append(res)
    # Memory-safe run: only one model is resident at a time.
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print(f'\nResults for {MODEL_NAME} (tuned threshold):')
for row in sorted(training_summary, key=lambda r: r['dev_macro_f1'], reverse=True): print(row)
with open(OUTPUT_DIR / 'comparison.json', 'w') as f: json.dump(training_summary, f, indent=2)

### Dev report of the best method

In [ ]:
winner = max(results, key=lambda k: results[k]['dev_macro_f1'])
pr = predictions[winner]; pred = (pr['dev_prob'] >= pr['threshold']).astype(int)
print(f'{MODEL_NAME} | {winner} | threshold {pr["threshold"]:.3f}')
print(classification_report(pr['ydev'], pred, digits=3, zero_division=0))
print('confusion matrix (rows = true, cols = predicted):\n', confusion_matrix(pr['ydev'], pred))

## 10. Section contribution analysis (dev-only ablation)
Reruns the winning model using only one already-selected chunk at a time. It tests *which document positions are informative* without leaking dev outcomes into training. The model was trained with all selected chunks, so these numbers are diagnostic, not controlled standalone training comparisons.

In [ ]:
winner = max(results, key=lambda k: results[k]['dev_macro_f1'])
model = make_model(winner); model = reload_best(model, OUTPUT_DIR / f'best_model{MODEL_OPTION}_{winner}_cjpe.pt').to(DEVICE)
ablation = []
for selection in (None, 'first', 'middle', 'last'):
    pp, yy, _ = predict_prob(model, dev_ds, selection)
    row = {'model': MODEL_NAME, 'chunks_used': selection or 'all', **metrics(yy, pp, .5)}
    ablation.append(row); print(row)
with open(OUTPUT_DIR / 'chunk_ablation.json', 'w') as f: json.dump(ablation, f, indent=2)
del model; gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## 11. Create exact competition JSON and ZIP
Exactly one `{id, pred}` per test record, in test order. Each prediction is a Python integer 0 or 1. The ZIP is named per model so runs of Models 4 / 6 / 7 do not overwrite each other.

In [ ]:
winner = max(results, key=lambda k: results[k]['dev_macro_f1'])
pr = predictions[winner]; th = pr['threshold']
submission = [{'id': str(case_id), 'pred': int(prob >= th)} for case_id, prob in zip(pr['test_ids'], pr['test_prob'])]
assert len(submission) == len(test) == len({r['id'] for r in submission})
assert [r['id'] for r in submission] == [r['id'] for r in test]
assert all(type(r['pred']) is int and r['pred'] in (0, 1) and set(r) == {'id', 'pred'} for r in submission)
output = OUTPUT_DIR / 'submission.json'
with open(output, 'w', encoding='utf-8') as f: json.dump(submission, f, ensure_ascii=False, indent=2)
zip_path = OUTPUT_DIR / f'submission_cjpe_model{MODEL_OPTION}_{winner}.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z: z.write(output, arcname='submission.json')
print(f'{MODEL_NAME} | winner: {winner} | threshold: {th:.3f} | test size: {len(submission)} | predicted 1s: {sum(r["pred"] for r in submission)}')
print('saved:', output, zip_path, '| sample:', submission[:3])
try:
    from IPython.display import display, FileLink
    display(FileLink(str(output))); display(FileLink(str(zip_path)))
except Exception: pass

## 12. Compare Models 4 / 6 / 7 (after you have run the notebook once per model)
Run the notebook with `MODEL_OPTION = 4`, then `6`, then `7` (same `REPRESENTATION`, `POOLING`, seed and splits). Each run writes `cjpe_p4_model<N>/comparison.json`; this cell collects whatever exists.

In [ ]:
rows = []
for fpath in sorted(OUTPUT_ROOT.glob('cjpe_p4_model*/comparison.json')):
    for r in json.load(open(fpath)): rows.append(r)
for r in sorted(rows, key=lambda r: r['dev_macro_f1'], reverse=True):
    print(f"{r.get('model','?'):8s} {r['method']:6s} {r.get('representation','?'):9s} {r.get('pooling','?'):9s} "
          f"dev macro-F1 {r['dev_macro_f1']:.4f} (@0.5: {r['dev_f1_05']:.4f}, thr {r['threshold']:.2f})")

**Knowledge distillation:** `kd` needs a full fine-tune first. If `'full'` is in `RUN_METHODS` it is run first and reused as the teacher; otherwise the teacher is trained automatically, or loaded from `KD_TEACHER_CKPT` (a `best_model<N>_full_cjpe.pt` from an earlier run). The student keeps `NUM_LAYERS // 2` layers and, like `full`, trains all its weights.

**Memory guidance (Kaggle T4, 16 GB):** each case sends up to `CHUNKS` windows of `MAX_LEN` tokens through the encoder at once. Start with `RUN_METHODS=['lora']` or `['lorma']`, `CHUNKS=4`, `MAX_LEN=256`; Model 6 (1,024d, 20 layers) is the heaviest — lower `CHUNKS`/`MAX_LEN` first if you hit OOM. `GRAD_CHECKPOINT=True` is already on for every method. LoRMA here uses Phase 3's multiplicative output-space definition, not a standardized PEFT-library adapter.